## Generate Parameterized Template Library

In [1]:
%pip install sympy


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""Controlled mathematical one-step pair generator. Requires sympy.

Labels are relative to a documented template rule, not a claim that arbitrary
mathematical relationships between expressions are absent.
"""
import json
import random
from pathlib import Path
import sympy as s

x = s.Symbol('x', real=True)
C = s.Symbol('C')


def nz(rng, lo=-10, hi=10):
    return rng.choice([v for v in range(lo, hi + 1) if v])


def draw(rng, template):
    q5_case = rng.choice(('two_roots', 'one_root', 'no_real_roots')) if template == 'Q5' else None
    while True:
        p = {k: nz(rng) for k in 'abcdef'}
        p.update(g=nz(rng), h=nz(rng))
        p['n'] = rng.randint(1, 10)
        p['m'] = rng.randint(1, 10)

        # Template-specific exponent and frequency ranges.
        if template in ('D1', 'I1'):
            p['n'] = rng.randint(2, 10)
            p['m'] = rng.randint(1, p['n']-1)
        elif template == 'D2':
            p['n'] = rng.randint(2, 10)
        elif template == 'I2':
            p['n'] = rng.randint(1, 10)
        elif template == 'Q6':
            p['a'] = rng.randint(1, 100)

        # Prevent positive answers and negative-hard answers are accidentaly equal.
        if template in ('S5', 'E8') and p['m'] == p['n']:
            continue
        if template == 'S5' and p['b'] == p['c']:
            continue
        if template == 'E6':
            if (
                p['a'] == 1
                or p['b'] == 1
                or p['a']*p['b'] == 1
                or p['e']*(p['a']-1) == 2*p['a']*p['b']*p['d']
                or p['e']*(1-p['b']) == 2*p['b']*p['d']
            ):
                continue
        if template == 'E7' and p['a']*p['c'] == -p['d']*p['f']:
            continue
        if template == 'E10' and p['a'] + p['c'] == p['a']*p['c']:
            continue
        if template == 'D3' and (p['a'] == p['c'] or p['b'] == p['d']):
            continue

        # Keep all sampled coefficients nonzero and ensure two distinct roots
        # in Q1/Q3; avoid accidental collisions in generated hard negatives.
        if template in ('Q1', 'Q3') and p['a']*p['d'] == p['b']*p['c']:
            continue
        if template == 'Q3' and p['a']*p['d'] + p['b']*p['c'] == 0:
            continue

        # Q5 deliberately covers two roots, one repeated root, and no real roots.
        if template == 'Q5':
            if q5_case == 'one_root':
                qa = nz(rng, -2, 2)
                root = nz(rng, -2, 2)
                p['a'] = qa
                p['b'] = -2*qa*root
                p['c'] = qa*root**2
            disc = p['b']**2 - 4*p['a']*p['c']
            if q5_case == 'two_roots' and disc <= 0:
                continue
            if q5_case == 'one_root' and disc != 0:
                continue
            if q5_case == 'no_real_roots' and disc >= 0:
                continue
            if q5_case in ('two_roots', 'one_root') and p['b'] == p['a']:
                continue
            if q5_case == 'one_root' and p['b'] == 2*p['a']:
                continue
            if q5_case == 'no_real_roots' and p['b'] == -2*p['a']:
                continue
            p['q5_case'] = q5_case

        # Q7 is restricted to real roots.
        if template == 'Q7' and (p['d']-p['c'])*p['a'] <= 0:
            continue

        # Template-specific exclusions prevent hard negatives from matching Pos.
        if template in ('D2', 'D5', 'D6', 'I2') and p['a'] == 1:
            continue
        if template in ('D4', 'I4', 'I5') and abs(p['a']) == 1:
            continue
        if template == 'L1' and (p['a'] == 1 or p['h'] == 1):
            continue
        if template in ('I6', 'L4', 'L6') and (
            abs(p['a']) == abs(p['b']) or abs(p['b']) == 1
        ):
            continue
        if template == 'L5' and abs(p['a']) == 1:
            continue
        if template == 'L8' and (p['a'] == 1 or abs(p['b']) == 1):
            continue
        if template == 'L7' and (abs(p['a']) == 1 or abs(p['a']) == abs(p['b']) or abs(p['b']) == 1):
            continue

        # L2 requires a defined original limit and defined hard negatives.
        if template == 'L2':
            numerator = p['a']*p['h'] + p['b']
            denominator = p['c']*p['h'] + p['d']
            if denominator == 0 or p['c'] + p['d'] == 0 or numerator == 0:
                continue
            pos = s.Rational(numerator, denominator)
            hard = (
                s.Rational(numerator, p['c'] + p['d']),
                s.Rational(p['a'] + p['b'], denominator),
                s.Rational(denominator, numerator),
            )
            if pos in hard:
                continue
        return p

In [3]:
def exprs(template, p):
    a,b,c,d,e,f = (s.Integer(p[k]) for k in 'abcdef')
    h = s.Integer(p['h'])
    n,m = p['n'],p['m']
    L = a*x+b
    R = c*x+d
    data = {
        'S1': a*x**2+b*x+c-d*x**2-e*x-f,
        'S2': a*s.exp(x)+b*x+c-d*s.exp(x)-e*x-f,
        'S3': s.Eq(a*x**2+b*x+c, d*x**2+e*x+f, evaluate=False),
        'S4': s.Eq(a*s.exp(x)+b*x+c, d*s.exp(x)+e*x+f, evaluate=False),
        'S5': s.Add(a*s.sin(n*x),b*s.cos(m*x),c*s.sin(n*x),d*s.cos(m*x),evaluate=False),
        'E1': s.Mul(L,R,evaluate=False),
        'E2': s.Pow(L,2,evaluate=False),
        'E3': s.Pow(L,3,evaluate=False),
        'E4': s.Mul(a*x**2+b*x+c,d*x+f,evaluate=False),
        'E5': s.Mul(L,R,e*x+f,evaluate=False),
        'E6': s.Mul(a,b*(c*x+d)+e,evaluate=False),
        'E7': s.Add(s.Mul(a,b*x+c,evaluate=False),s.Mul(d,e*x+f,evaluate=False),evaluate=False),
        'E8': s.Mul(a*s.exp(m*x)+b,c*s.exp(n*x)+d,evaluate=False),
        'E9': s.Mul(a*s.sin(n*x)+b,c*s.sin(n*x)+d,evaluate=False),
        'E10': s.Mul(a*s.sin(n*x)+b,c*s.cos(m*x)+d,evaluate=False),
        'Q1': s.Eq(s.Mul(L,R,evaluate=False),0,evaluate=False),
        'Q2': s.Eq(s.Pow(L,2,evaluate=False),0,evaluate=False),
        'Q3': s.Eq(a*c*x**2+(a*d+b*c)*x+b*d,0,evaluate=False),
        'Q4': s.Eq(a*a*x**2+2*a*b*x+b*b,0,evaluate=False),
        'Q5': s.Eq(a*x**2+b*x+c,0,evaluate=False),
        'Q6': s.Eq(x**2-a**2,0,evaluate=False),
        'Q7': s.Eq(a*(x-b)**2+c,d,evaluate=False),
        'D1': a*x**n+c*x**m+b,
        'D2': s.Pow(L,n,evaluate=False),
        'D3': s.Mul(L,R,evaluate=False),
        'D4': s.sin(L),
        'D5': s.exp(L),
        'D6': s.log(L),
        'I1': a*x**n+c*x**m+b,
        'I2': s.Pow(L,n,evaluate=False),
        'I3': s.Mul(L,R,evaluate=False),
        'I4': s.sin(L),
        'I5': s.exp(L),
        'I6': a/(b*x),
        'L1': s.Limit(a*x**2+b*x+c,x,h),
        'L2': s.Limit((a*x+b)/(c*x+d),x,h),
        'L3': s.Tuple(s.Limit(b/(a*x),x,0,dir='+'),s.Limit(b/(a*x),x,0,dir='-')),
        'L4': s.Limit((s.exp(a*x)-1)/(b*x),x,0),
        'L5': s.Limit(a*(x**2-h**2)/(x-h),x,h),
        'L6': s.Limit(s.sin(a*x)/(b*x),x,0),
        'L7': s.Limit(s.sin(a*x)/s.sin(b*x),x,0),
        'L8': s.Limit((1-s.cos(a*x))/(b*x**2),x,0),
    }
    return data[template]

In [4]:
def display_atom(item):
    item = s.sympify(item)
    if isinstance(item, s.Number) and item.is_negative:
        return s.Symbol(r'\left(' + s.latex(item) + r'\right)')
    return item


def hold_add(*items):
    return s.Add(*(display_atom(item) for item in items), evaluate=False)


def hold_mul(*items):
    return s.Mul(*(display_atom(item) for item in items), evaluate=False)


def hold_neg(item):
    return s.Mul(-1, display_atom(item), evaluate=False)


def hold_sub(left, right):
    return s.Add(display_atom(left), hold_neg(right), evaluate=False)


def displayed_candidate(expr, display_expr=None):
    return {
        'expr': expr,
        'display_expr': expr if display_expr is None else display_expr
    }


def candidate_expr(item):
    return item['expr'] if isinstance(item, dict) else item


def candidate_latex(item):
    if isinstance(item, dict):
        return s.latex(item['display_expr'], order='none')
    return s.latex(item)


def response_latex(template, item, use_display=False):
    text = candidate_latex(item) if use_display else s.latex(item)
    if template.startswith('I'):
        text += r' + C'
    return text


def candidate_solutions(template, p):
    """Return positive and hard-negative candidate expressions.

    For equations, use equivalent equations (not isolated individual roots).
    For integrals, represent antiderivatives without +C; equality is modulo C.
    """
    a,b,c,d,e,f = (s.Integer(p[k]) for k in 'abcdef')
    h = s.Integer(p['h'])
    n,m = p['n'],p['m']
    L,R = a*x+b,c*x+d
    eq = lambda z: s.Eq(z,0,evaluate=False)
    if template == 'S1':
        v=(a-d)*x**2+(b-e)*x+c-f
        display2=hold_add(hold_mul(hold_sub(a,d),x**2),hold_mul(hold_sub(b,e),x),hold_sub(c,f))
        return [displayed_candidate(v),displayed_candidate(v,display2)], [v+2*d*x**2, v+2*e*x, v+2*f]
    if template == 'S2':
        v=(a-d)*s.exp(x)+(b-e)*x+c-f
        display2=hold_add(hold_mul(hold_sub(a,d),s.exp(x)),hold_mul(b,x),hold_neg(hold_mul(e,x)),c,hold_neg(f))
        return [displayed_candidate(v),displayed_candidate(v,display2)], [v+2*d*s.exp(x),v+2*e*x,v+2*f]
    if template == 'S3':
        v=(a-d)*x**2+(b-e)*x+c-f
        return [eq(v),s.Eq((a-d)*x**2+(b-e)*x,f-c,evaluate=False)], [eq(v+2*d*x**2),eq(v+2*e*x),eq(v+2*f)]
    if template == 'S4':
        v=(a-d)*s.exp(x)+(b-e)*x+c-f
        pos2=s.Eq((a-d)*s.exp(x)+(b-e)*x,f-c,evaluate=False)
        display2=s.Eq(hold_add(hold_mul(hold_sub(a,d),s.exp(x)),hold_mul(hold_sub(b,e),x)),hold_sub(f,c),evaluate=False)
        return [displayed_candidate(eq(v)),displayed_candidate(pos2,display2)], [eq(v+2*d*s.exp(x)),eq(v+2*e*x),eq(v+2*f)]
    if template == 'S5':
        v=(a+c)*s.sin(n*x)+(b+d)*s.cos(m*x)
        return [v], [(a-c)*s.sin(n*x)+(b+d)*s.cos(m*x),(a+b)*s.sin(n*x)+(c+d)*s.cos(m*x),(a+c)*s.sin(m*x)+(b+d)*s.cos(n*x)]
    if template == 'E1':
        v=s.expand(L*R)
        return [v, s.Add(a*c*x**2,a*d*x,b*c*x,b*d,evaluate=False)], [a*c*x**2+(a*d-b*c)*x+b*d,v-2*b*d,(a+c)*x**2+(b+d)*x+b*d]
    if template == 'E2':
        v=s.expand(L**2)
        return [v,s.Add(a*a*x*x,a*b*x,a*b*x,b*b,evaluate=False)], [a*a*x**2+a*b*x+b*b,a*a*x**2-2*a*b*x+b*b,a*a*x**2+2*a*b*x-b*b]
    if template == 'E3':
        v=s.expand(L**3)
        return [v,s.Add(a**3*x**3,3*a*a*b*x**2,a*b*b*x,2*a*b*b*x,b**3,evaluate=False)], [a**3*x**3+3*a*a*b*x**2+3*a*b*b+b**3,a**3*x**3+2*a*a*b*x**2+3*a*b*b*x+b**3,v-2*b**3]
    if template == 'E4':
        v=s.expand((a*x**2+b*x+c)*(d*x+f))
        return [v], [a*d*x**3+(a*f+b*d)*x**2+(b*f-c*d)*x+c*f,a*d*x**3+a*f*x**2+(b*f+b*d)*x+c*f,v-2*c*f]
    if template == 'E5':
        v=s.expand(L*R*(e*x+f))
        pos2=s.Add(a*c*e*x**3,a*c*f*x**2,e*(a*d+b*c)*x**2,f*(a*d+b*c)*x,b*d*e*x,b*d*f,evaluate=False)
        return [v,pos2], [v-2*b*d*f,(a+c+e)*x**3+(a*c+c*e+a*e)*x**2+(c*e+b*c+b*e)*x-b*d*f]
    if template == 'E6':
        v=a*b*c*x+a*b*d+a*e
        return [v], [a*b*c*x-a*b*d+a*e,a*b*c*x+a*b*d+e,a*b*c*x+a*b*d+a*b*e]
    if template == 'E7':
        v=(a*b+d*e)*x+a*c+d*f
        return [v,s.Add(a*b*x,d*e*x,a*c,d*f,evaluate=False)], [a*b*x+d*e*x+c+f,(a*b-d*e)*x+a*c+d*f,v-2*d*f]
    if template == 'E8':
        v=s.expand((a*s.exp(m*x)+b)*(c*s.exp(n*x)+d))
        pos2=s.Add(a*c*s.exp((m+n)*x),a*d*s.exp(m*x),b*c*s.exp(n*x),b*d,evaluate=False)
        display2=hold_add(hold_mul(a,c,s.exp(hold_mul(hold_add(s.Integer(m),s.Integer(n)),x))),hold_mul(a,d,s.exp(hold_mul(s.Integer(m),x))),hold_mul(b,c,s.exp(hold_mul(s.Integer(n),x))),hold_mul(b,d))
        return [displayed_candidate(v),displayed_candidate(pos2,display2)], [a*c*s.exp(m*n*x)+a*d*s.exp(m*x)+b*c*s.exp(n*x)+b*d,a*s.exp(m*x)+b*c*s.exp(n*x)+d,(a+c)*s.exp((m+n)*x)+a*d*s.exp(m*x)+b*c*s.exp(n*x)+b*d,a*c*s.exp((m+n)*x)+(a*d+b*c)*s.exp(m*x)+b*d]
    if template == 'E9':
        v=a*c*s.sin(n*x)**2+(a*d+b*c)*s.sin(n*x)+b*d
        return [v], [a*c*s.sin(2*n*x)+(a*d+b*c)*s.sin(n*x)+b*d,a*c*s.sin(n*x)**2+(a*d-b*c)*s.sin(n*x)+b*d,a*c*s.sin(n*x**2)+(a*d+b*c)*s.sin(n*x)+b*d]
    if template == 'E10':
        v=a*c*s.sin(n*x)*s.cos(m*x)+a*d*s.sin(n*x)+b*c*s.cos(m*x)+b*d
        return [v], [(a+c)*s.sin(n*x)*s.cos(m*x)+a*d*s.sin(n*x)+b*c*s.cos(m*x)+b*d,a*c*s.sin(n*x)*s.cos(m*x)+a*d*s.sin(n*x)-b*c*s.cos(m*x)+b*d,a*c*s.sin((n+m)*x)+a*d*s.sin(n*x)+b*c*s.cos(m*x)+b*d,a*c*s.sin(n*x)*s.cos(m*x)+(a*d+b*c)*s.cos(m*x)*s.sin(n*x)+b*d,a*c*s.sin(n*x)*s.cos(m*x)+a*d*s.sin(n*x)-b*c*s.cos(m*x)-b*d]
    if template == 'Q1':
        return [eq(s.expand(L*R)), s.Or(eq(L),eq(R),evaluate=False)], [s.Or(eq(a*x-b),eq(c*x-d),evaluate=False),eq(s.expand((a*x-b)*(c*x-d)))]
    if template == 'Q2':
        return [eq(L),s.Eq(x,-b/a,evaluate=False)], [eq(a*x-b),s.Eq(x,b/a,evaluate=False)]
    if template == 'Q3':
        return [eq(L*R),s.Or(s.Eq(x,-b/a,evaluate=False),s.Eq(x,-d/c,evaluate=False),evaluate=False)], [eq((a*x-b)*(c*x-d)),s.Or(s.Eq(x,b/a,evaluate=False),s.Eq(x,d/c,evaluate=False),evaluate=False)]
    if template == 'Q4':
        return [eq(L**2),eq(L),s.Eq(x,-b/a,evaluate=False)], [eq((a*x-b)**2),eq(a*x-b),s.Eq(x,b/a,evaluate=False)]
    if template == 'Q5':
        disc=b**2-4*a*c
        if disc > 0:
            root1=(-b+s.sqrt(disc))/(2*a)
            root2=(-b-s.sqrt(disc))/(2*a)
            pos=s.Or(s.Eq(x,root1,evaluate=False),s.Eq(x,root2,evaluate=False),evaluate=False)
            wrong_disc=b**2+4*a*c
            hard=[s.Or(s.Eq(x,(b+s.sqrt(disc))/(2*a),evaluate=False),s.Eq(x,(b-s.sqrt(disc))/(2*a),evaluate=False),evaluate=False),s.Or(s.Eq(x,(-b+s.sqrt(wrong_disc))/(2*a),evaluate=False),s.Eq(x,(-b-s.sqrt(wrong_disc))/(2*a),evaluate=False),evaluate=False),s.Or(s.Eq(x,root1+1,evaluate=False),s.Eq(x,root2+1,evaluate=False),evaluate=False)]
        elif disc == 0:
            root=-b/(2*a)
            pos=s.Eq(x,root,evaluate=False)
            hard=[s.Eq(x,b/(2*a),evaluate=False),s.Eq(x,root+1,evaluate=False),s.Eq(x,0,evaluate=False)]
        else:
            pos=s.EmptySet
            hard=[s.Eq(x,0,evaluate=False),s.Eq(x,1,evaluate=False),s.Eq(x,-b/(2*a),evaluate=False)]
        return [pos], hard
    if template == 'Q6':
        return [eq((x-a)*(x+a)),s.Or(s.Eq(x,a,evaluate=False),s.Eq(x,-a,evaluate=False),evaluate=False)], [eq((x-a)**2),s.Eq(x,s.Abs(a),evaluate=False)]
    if template == 'Q7':
        rad=(d-c)/a
        roots=s.Or(s.Eq(x,b+s.sqrt(rad),evaluate=False),s.Eq(x,b-s.sqrt(rad),evaluate=False),evaluate=False)
        return [s.Eq((x-b)**2,rad,evaluate=False),roots], [s.Eq((x-b)**2,(d+c)/a,evaluate=False),s.Or(s.Eq(x,b+s.sqrt(d-c)/a,evaluate=False),s.Eq(x,b-s.sqrt(d-c)/a,evaluate=False),evaluate=False),s.Or(s.Eq(x,-b+s.sqrt(d-c)/a,evaluate=False),s.Eq(x,-b-s.sqrt(d-c)/a,evaluate=False),evaluate=False)]
    if template == 'D1':
        v=a*n*x**(n-1)+c*m*x**(m-1)
        return [v,s.Derivative(a*x**n,x)+s.Derivative(c*x**m,x)], [a*x**(n-1)+c*x**(m-1),a*(n-1)*x**(n-1)+c*(m-1)*x**(m-1),a*n*x**n+c*m*x**m]
    if template == 'D2':
        return [n*a*L**(n-1),s.Mul(n,L**(n-1),a,evaluate=False)], [n*L**(n-1),n*a*L**n,n*a*(a*x-b)**(n-1)]
    if template == 'D3':
        pos1=a*R+c*L
        pos2=2*a*c*x+a*d+b*c
        display1=hold_add(hold_mul(a,R),hold_mul(c,L))
        return [displayed_candidate(pos1,display1),displayed_candidate(pos2)], [a*R-c*L,a*c*x+a*d+b*c,2*a*c*x+a*b+c*d]
    if template == 'D4':
        return [a*s.cos(L),s.Mul(s.cos(L),a,evaluate=False)], [s.cos(L),-a*s.cos(L),a*s.cos(a*x-b)]
    if template == 'D5':
        return [a*s.exp(L),s.Mul(s.exp(L),a,evaluate=False)], [s.exp(L),-a*s.exp(L),a*s.exp(a*x-b)]
    if template == 'D6':
        return [a/L,s.Mul(1/L,a,evaluate=False)], [1/L,1/x,a*s.log(L)]
    if template == 'I1':
        v=a*x**(n+1)/(n+1)+c*x**(m+1)/(m+1)+b*x
        return [v,s.Integral(a*x**n,x)+s.Integral(c*x**m,x)+s.Integral(b,x)], [a*x**(n+1)/n+c*x**(m+1)/m+b*x,a*x**(n+1)/(n+1)+c*x**(m+1)/(m+1)+b,a*x**n/(n+1)+c*x**m/(m+1)+b*x]
    if template == 'I2':
        return [L**(n+1)/(a*(n+1))], [L**(n+1)/(n+1),L**n/(a*n),(a*x-b)**(n+1)/(a*(n+1))]
    if template == 'I3':
        v=a*c*x**3/3+(a*d+b*c)*x**2/2+b*d*x
        return [v,s.Integral(s.expand(L*R),x)], [a*c*x**3/2+(a*d+b*c)*x**2/2+b*d*x,a*c*x**3/3+(a*d+b*c)*x**2+b*d*x,a*c*x**3/3+(a*d+b*c)*x**2/2+b*d*x**2]
    if template == 'I4':
        return [-s.cos(L)/a], [s.cos(L)/a,-a*s.cos(L),-s.cos(a*x-b)/a]
    if template == 'I5':
        return [s.exp(L)/a], [a*s.exp(L),-s.exp(L)/a,s.exp(a*x-b)/a]
    if template == 'I6':
        v=a*s.log(s.Abs(x))/b
        return [v], [b*s.log(s.Abs(x))/a,a*b*s.log(s.Abs(x)),a*s.log(s.Abs(x))]
    if template == 'L1':
        return [a*h**2+b*h+c], [a*h+b*h+c,(a*h)**2+b*h+c,a*h**2+b+c]
    if template == 'L2':
        v=s.Rational(a*h+b,c*h+d)
        return [v], [s.Rational(a*h+b,c+d),s.Rational(a+b,c*h+d),s.Rational(c*h+d,a*h+b)]
    if template == 'L3':
        right=s.Limit(b/(a*x),x,0,dir='+')
        left=s.Limit(b/(a*x),x,0,dir='-')
        right_value=s.oo if a*b > 0 else -s.oo
        left_value=-right_value
        return [s.Eq(right,right_value,evaluate=False),s.Eq(left,left_value,evaluate=False)], [s.Eq(right,left_value,evaluate=False),s.Eq(left,right_value,evaluate=False),s.Eq(right,0,evaluate=False)]
    if template == 'L4':
        return [a/b], [b/a,a*b,a]
    if template == 'L5':
        return [2*a*h], [a*h,2*h/a,2*a+h]
    if template == 'L6':
        return [a/b], [b/a,a*b,a]
    if template == 'L7':
        return [a/b], [b/a,s.Integer(1),a*b]
    if template == 'L8':
        return [a**2/(2*b)], [a/(2*b),a**2*b/2,a**2/2]
    raise ValueError(template)

In [5]:
def is_equivalent(template, original, candidate):
    def same_expression(left, right):
        difference=left-right
        if difference == 0:
            return True
        for value in (0, 1, 2, -1):
            try:
                result=s.N(difference.subs(x,value))
                if result.is_number and result.is_finite:
                    if abs(complex(result)) > 1e-9:
                        return False
            except (TypeError, ValueError):
                pass
        return s.simplify(difference)==0
    if template.startswith(('S','E')):
        if isinstance(original,s.Equality):
            if not isinstance(candidate,s.Equality):
                return False
            # candidate and original are equivalent if their difference is zero.
            u=s.expand(original.lhs-original.rhs)
            v=s.expand(candidate.lhs-candidate.rhs)
            return same_expression(u,v) or same_expression(u,-v)
        if isinstance(candidate,(s.Equality,s.logic.boolalg.Boolean)):
            return False
        return same_expression(original,candidate)
    if template.startswith('Q'):
        try:
            # Real solution sets: avoids treating one root of a two-root
            # quadratic as an equivalent complete solution.
            def roots(formula):
                if isinstance(formula,s.Or):
                    result=s.EmptySet
                    for part in formula.args: result=result.union(roots(part))
                    return result
                if isinstance(formula,s.Equality):
                    if formula.lhs.func == s.atan and formula.rhs.is_real:
                        if not (-s.pi/2 < formula.rhs < s.pi/2):
                            return s.EmptySet
                    if formula.rhs.func == s.atan and formula.lhs.is_real:
                        if not (-s.pi/2 < formula.lhs < s.pi/2):
                            return s.EmptySet
                    return s.solveset(formula.lhs-formula.rhs,x,domain=s.S.Reals)
                return s.EmptySet
            base=roots(original)
            if candidate == s.EmptySet:
                return base == s.EmptySet
            if candidate in (s.true,s.false):
                return base == s.EmptySet
            if not isinstance(candidate,(s.Equality,s.Or,s.FiniteSet)):
                return False
            if isinstance(base,s.FiniteSet):
                parts=candidate.args if isinstance(candidate,s.Or) else [candidate]
                # A candidate that fails at any original root cannot be
                # solution-set equivalent. This cheaply rejects transcendental
                # easy negatives without attempting transcendental solveset.
                for root in base:
                    checks=[s.N((part.lhs-part.rhs).subs(x,root)) for part in parts]
                    if all(abs(complex(z))>1e-9 for z in checks):
                        return False
            return base==roots(candidate)
        except (AttributeError, NotImplementedError, ValueError, TypeError):
            return False
    if template.startswith('D'):
        if isinstance(candidate,s.logic.boolalg.Boolean):
            return False
        return same_expression(s.diff(original,x),candidate.doit())
    if template == 'I6':
        if isinstance(candidate,s.logic.boolalg.Boolean):
            return False
        coefficient=s.simplify(original*x)
        return same_expression(candidate,coefficient*s.log(s.Abs(x)))
    if template.startswith('I'):
        if isinstance(candidate,s.logic.boolalg.Boolean):
            return False
        return same_expression(s.diff(candidate.doit(),x),original)
    if template.startswith('L'):
        if isinstance(original,s.Tuple):
            if not isinstance(candidate,s.Equality):
                return False
            expected=candidate.lhs.doit()
            actual=candidate.rhs.doit()
        else:
            if isinstance(candidate,s.logic.boolalg.Boolean):
                return False
            expected=original.doit()
            actual=candidate.doit()
        return expected==actual or same_expression(expected,actual)
    return False

In [6]:
def render_problem(template, original, p):
    if template == "S1":
        return (f"({p['a']})x^2+({p['b']})x+({p['c']})"
                f"-({p['d']})x^2-({p['e']})x-({p['f']})")
    if template == "S2":
        return (f"({p['a']})e^x+({p['b']})x+({p['c']})"
                f"-({p['d']})e^x-({p['e']})x-({p['f']})")
    if template.startswith('D'):
        return r'\frac{d}{dx}\left('+s.latex(original)+r'\right)'
    if template.startswith('I'):
        return r'\int\left('+s.latex(original)+r'\right)\,dx'
    return s.latex(original)

In [7]:
G,H,P,Q,R,N = s.symbols('G H P Q R N', nonzero=True)

# X01-X40 from Neg_E_Inventory.xlsx.
base_negatives = [
    G*s.log(P*x+Q)+H,
    G*s.log(x+H,P)+Q,
    s.log(G*x+H)+s.log(P*x+Q),
    s.log(H-x,G),
    G**s.log(P*x+H),
    (G*x+H)/(P*x+Q),
    G/(x+H)+P/(x+Q),
    (G*x**2+H)/(P*x+Q),
    1/(G*x+H),
    (x+G)/(x**2+H),
    G*s.sqrt(x+H)+P,
    s.sqrt(G*x**2+H),
    G*(P*x+H)**s.Rational(1,3),
    (G+s.sqrt(x+H))/(P+Q*x),
    s.sqrt(G*x+H)-s.sqrt(P*x+Q),
    G*s.Abs(x-H)+P,
    s.Abs(G*x+H)/(P+Q),
    s.Abs(x**2-H),
    s.Max(G*x+H,P*x+Q),
    s.sign(G*x+H),
    G*s.factorial(x+H)+P,
    s.binomial(x+H,P),
    s.factorial(x+P)/s.factorial(x+Q),
    G*s.factorial(N)+H,
    s.factorial(N+P)/s.factorial(N),
    G*s.sinh(P*x+H),
    G*s.cosh(P*x+H)+Q,
    s.tanh(G*x+H),
    G*s.sinh(P*x)+H*s.cosh(Q*x),
    s.sech(G*x+H),
    G*s.tan(P*x+H),
    G*s.sec(P*x)+H,
    s.sin(G*x+H)*s.cos(P*x+Q),
    s.atan(G*x+H),
    G*s.sin(P*x**2+H),
    G*s.exp(P*x+H),
    2**(P*x+H)+Q,
    s.exp(G*x**2+H),
    G*s.exp(-P*x)+H*s.exp(Q*x),
    s.exp(s.exp(G*x+H))
]

# R01-R20 from Neg_E_Inventory.xlsx.
relation_negatives = [
    s.Eq(G*s.exp(x)+H,0,evaluate=False),
    s.Eq(G*s.exp(P*x)+H,Q,evaluate=False),
    s.Eq(2**(G*x+H),P,evaluate=False),
    s.Eq(s.log(G*x+H),P,evaluate=False),
    s.Eq(s.log(x+H,G),P,evaluate=False),
    s.Eq(s.log(x**2+H),P,evaluate=False),
    s.Eq(s.sin(G*x+H),0,evaluate=False),
    s.Eq(s.cos(G*x+H),0,evaluate=False),
    s.Eq(s.tan(G*x+H),P,evaluate=False),
    s.Eq(s.sec(G*x+H),P,evaluate=False),
    s.Eq(s.sqrt(G*x+H),P,evaluate=False),
    s.Eq((G*x+H)**s.Rational(1,3),P,evaluate=False),
    s.Eq(1/(G*x+H),P,evaluate=False),
    s.Eq((G*x+H)/(P*x+Q),R,evaluate=False),
    s.Eq(s.Abs(G*x+H),P,evaluate=False),
    s.Eq(s.Abs(x-G)+s.Abs(x-H),P,evaluate=False),
    s.Eq(s.floor(G*x+H),P,evaluate=False),
    s.Eq(s.cosh(G*x+H),P,evaluate=False),
    s.Eq(s.atan(G*x+H),P,evaluate=False),
    s.Eq(s.sign(G*x+H),1,evaluate=False)
]

# X41-X87 from Neg_E_Inventory.xlsx.
integration_negatives = [
    G*s.sin(P*x+H)+C,
    G*s.cos(P*x+H)+Q+C,
    G*s.tan(P*x+H)+C,
    G*x**2+H*x+Q+C,
    G*x**3+H*x**2+P*x+Q+C,
    (G*x+H)*(P*x+Q)+C,
    G*s.exp(P*x+H)+C,
    G*s.exp(P*x)+H*s.exp(Q*x)+C,
    G*x*s.exp(P*x)+C,
    G*x**2/2+H*x+C
]
integration_negatives += [item+C for item in base_negatives[:30]]
integration_negatives += [
    item+C
    for item in base_negatives[31:35]+base_negatives[37:40]
]

easy_negative_dict = {
    'simplification': [],
    'expansion': [],
    'quadratic_equation': [],
    'differentiation': [],
    'integration': integration_negatives,
    'limits': []
}

for neg_id, item in enumerate(base_negatives,start=1):
    easy_negative_dict['expansion'].append(item)
    easy_negative_dict['limits'].append(item)
    if neg_id not in (34,35,36,37,39):
        easy_negative_dict['differentiation'].append(item)
    if neg_id not in (25,26,27,28,29,30):
        easy_negative_dict['simplification'].append(item)

for item in relation_negatives:
    easy_negative_dict['simplification'].append(item)
    easy_negative_dict['quadratic_equation'].append(item)
    easy_negative_dict['limits'].append(item)

In [8]:
def substitute_easy_negative(item, substitutions):
    if isinstance(item, s.Equality):
        return s.Eq(
            item.lhs.subs(substitutions),
            item.rhs.subs(substitutions),
            evaluate=False
        )
    return item.subs(substitutions)

In [9]:
def generate_neg_hard(rng, template, p, original):
    used_parameters = {tuple(sorted(p.items()))}
    hard_dict = {}

    def add_hard(parameter):
        _, candidates = candidate_solutions(template, parameter)

        for item in candidates:
            if not is_equivalent(template, original, item):
                hard_dict[s.latex(item)] = item

    # 先加入当前题目原本的 Neg-H
    add_hard(p)
    original_keys = list(hard_dict)

    # 先额外生成2–3组参数
    groups_needed = rng.randint(2, 3)

    while groups_needed > 0 or len(hard_dict) < 8:
        new_p = draw(rng, template)
        parameter_key = tuple(sorted(new_p.items()))

        if parameter_key in used_parameters:
            continue

        used_parameters.add(parameter_key)
        add_hard(new_p)
        groups_needed -= 1

    # 超过10个时，保留原始 Neg-H，只随机选择新增的
    if len(hard_dict) > 10:
        original_items = [
            hard_dict[key] for key in original_keys
        ]
        extra = [
            item for key, item in hard_dict.items()
            if key not in original_keys
        ]
        hard = original_items + rng.sample(
            extra,
            k=10-len(original_items)
        )
    else:
        hard = list(hard_dict.values())

    return hard

In [10]:
def generate_dataset(
    n_per_template,
    seed,
    selected_templates=None
):
    rng = random.Random(seed)

    categories = {
        "simplification": ["S1", "S2", "S3", "S4", "S5"],
        "expansion": ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9", "E10"],
        "quadratic_equation": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"],
        "differentiation": ["D1", "D2", "D3", "D4", "D5", "D6"],
        "integration": ["I1", "I2", "I3", "I4", "I5", "I6"],
        "limits": ["L1", "L2", "L3", "L4", "L5", "L6", "L7", "L8"]
    }

    records = []
    seen_parameter_lists = {
        template: set()
        for templates in categories.values()
        for template in templates
    }
    seen_problems = {
        template: set()
        for templates in categories.values()
        for template in templates
    }
    stats = {
        "skipped_invalid": 0,
        "skipped_easy_equivalent": 0,
        "skipped_duplicate_parameters": 0,
        "skipped_duplicate_problem": 0
    }

    for category, templates in categories.items():
        for template in templates:
            if selected_templates is not None and template not in selected_templates:
                continue
            while sum(
                record["template"] == template
                for record in records
            ) < n_per_template:
                p = draw(rng, template)
                parameter_list = tuple(sorted(p.items()))
                if parameter_list in seen_parameter_lists[template]:
                    stats["skipped_duplicate_parameters"] += 1
                    continue

                original = exprs(template, p)
                problem = render_problem(template, original, p)
                if problem in seen_problems[template]:
                    stats["skipped_duplicate_problem"] += 1
                    continue

                pos, _ = candidate_solutions(template, p)
                hard = generate_neg_hard(rng, template, p, original)

                pos_checks = [
                    is_equivalent(template, original, candidate_expr(item))
                    for item in pos
                ]
                
                if not all(pos_checks):
                    stats["skipped_invalid"] += 1
                    continue

                easy_subs = {
                    G: abs(p["g"])+1,
                    H: p["h"],
                    P: abs(p["a"])+1,
                    Q: abs(p["b"])+1,
                    R: p["c"],
                    N: p["n"]
                }
                easy_pool = [
                    substitute_easy_negative(item, easy_subs)
                    for item in easy_negative_dict[category]
                ]
                easy = rng.sample(
                    easy_pool,
                    k=20
                )
                easy_checks = [
                    item not in (s.true, s.false)
                    and not is_equivalent(
                        template, original, item
                    )
                    for item in easy
                ]
                if not all(easy_checks):
                    stats["skipped_easy_equivalent"] += 1
                    continue

                seen_parameter_lists[template].add(parameter_list)
                seen_problems[template].add(problem)
                records.append({
                    "index": len(records),
                    "category": category,
                    "template": template,
                    "problem": problem,
                    "responses": {
                        "positive": [
                            response_latex(template, item, use_display=True)
                            for item in pos
                        ],
                        "negative_hard": [
                            response_latex(template, item)
                            for item in hard
                        ],
                        "negative_easy": [
                            s.latex(item) for item in easy
                        ]
                    }
                })

    return records, stats

In [11]:
def save_pretty_json(records, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with path.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            records,
            f,
            ensure_ascii=False,
            indent=2
        )

In [ ]:
seed = 45
selected_templates = None

records, stats = generate_dataset(
    n_per_template=50,
    seed=seed,
    selected_templates=selected_templates
)

save_pretty_json(
    records,
    f"/Users/yihua/Desktop/Catherine/UCSB/Intern Related/Gao Research/2026/10.17 - 10.23/refined_positive_templates_{seed}_pretty.json"
)

print("Pretty JSON saved")
print("Total records:", len(records))
print("Stats:", stats)
print("Pretty JSON saved")